<a href="https://colab.research.google.com/github/Rishii077/AI-Lab-Assignments/blob/main/Experiment_4_SQL_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai

In [2]:
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

print("Gemini connected successfully!")

Gemini connected successfully!


In [3]:
import sqlite3

conn = sqlite3.connect("college.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    marks INTEGER
)
""")

cursor.execute("DELETE FROM students")

students = [
    (1, "Rahul", "CSE", 85),
    (2, "Priya", "ECE", 91),
    (3, "Arjun", "CSE", 78),
    (4, "Sneha", "IT", 88),
    (5, "Kiran", "ECE", 95)
]

cursor.executemany(
    "INSERT INTO students VALUES (?, ?, ?, ?)",
    students
)

conn.commit()

print("Database created successfully!")

Database created successfully!


In [4]:
def database_tool(sql_query):

    try:
        cursor.execute(sql_query)
        results = cursor.fetchall()

        return results

    except Exception as e:
        return f"Database error: {e}"

In [5]:
print(database_tool("SELECT * FROM students"))

[(1, 'Rahul', 'CSE', 85), (2, 'Priya', 'ECE', 91), (3, 'Arjun', 'CSE', 78), (4, 'Sneha', 'IT', 88), (5, 'Kiran', 'ECE', 95)]


In [6]:
def sql_agent(question):

    prompt = f"""
You are an SQL Agent.

You have access to a SQLite database with this table:

students(
    id,
    name,
    department,
    marks
)

Your task is to convert the user's question into a SQL query.

Return ONLY the SQL query.
Do not provide explanations.
Do not use markdown code blocks.

User Question:
{question}
"""

    interaction = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    sql_query = interaction.output_text.strip()

    # Remove markdown code blocks if Gemini adds them
    sql_query = sql_query.replace("```sql", "").replace("```", "").strip()

    # Agent uses the database tool
    results = database_tool(sql_query)

    print("User Question:")
    print(question)

    print("\nGenerated SQL:")
    print(sql_query)

    print("\nTool Output:")
    print(results)

In [7]:
sql_agent("Which students scored more than 90 marks?")

User Question:
Which students scored more than 90 marks?

Generated SQL:
SELECT name FROM students WHERE marks > 90;

Tool Output:
[('Priya',), ('Kiran',)]


In [8]:
sql_agent("Which students are from the CSE department?")

User Question:
Which students are from the CSE department?

Generated SQL:
SELECT * FROM students WHERE department = 'CSE';

Tool Output:
[(1, 'Rahul', 'CSE', 85), (3, 'Arjun', 'CSE', 78)]


In [9]:
sql_agent("What is the average marks of all students?")

User Question:
What is the average marks of all students?

Generated SQL:
SELECT AVG(marks) FROM students;

Tool Output:
[(87.4,)]
